В этом ноутбуке мы будем делать регрессию для таргета СC50

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
df = pd.read_pickle('/content/dataset_after_eda.pkl')
df = df.drop_duplicates()
df.shape

(958, 211)

In [5]:
y = np.log10(df['CC50, mM'])
X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape[0])
print(X_test.shape[0])

766
192


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("regressor", LinearRegression())]),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Ridge())]),
    "Lasso Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Lasso(alpha=0.1))]),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False) # verbose=0, чтобы не спамить логами
}

baseline_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_test)

    mae_log = mean_absolute_error(y_test, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
    r2 = r2_score(y_test, y_pred_log)

    y_test_real = 10 ** y_test
    y_pred_real = 10 ** y_pred_log

    mae_real = mean_absolute_error(y_test_real, y_pred_real)

    baseline_results.append({
        "Model": name,
        "MAE ": round(mae_log, 4),
        "RMSE ": round(rmse_log, 4),
        "MAE (реальный, mM)": round(mae_real, 4),
        "R2 Score": round(r2, 4)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="R2 Score", ascending=False)
df_baseline

,Model,MAE,RMSE,"MAE (реальный, mM)",R2 Score
3,Random Forest,0.3386,0.4894,271.6748,0.5162
5,CatBoost,0.3400,0.4964,267.9647,0.5023
4,Gradient Boosting,0.3600,0.5029,290.2147,0.4891
1,Ridge Regression,0.4027,0.5454,329.0804,0.3993
0,Linear Regression,0.4227,0.5724,393.9844,0.3384
2,Lasso Regression,0.5309,0.6542,433.0189,0.1356


Тут уже лучше себя показывает случайный лес, среди базовых вариантов RF выглядит наиболее перспективным

In [12]:
grid = {
    'n_estimators': [300, 400],
    'max_depth': [12, 15],
    'min_samples_split': [3, 4],
    'min_samples_leaf': [2]
}

rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Random Forest")
rf_search.fit(X_train, y_train)

print(rf_search.best_params_)
best_rf_model = rf_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_rf))
print("MAE: ", mean_absolute_error(y_test, y_pred_best_rf))
print("MSE: ", mean_squared_error(y_test, y_pred_best_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best_rf)))

Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits
{'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 400}
R2: 0.5197265535867324
MAE:  0.3395063134855194
MSE:  0.23779886927044788
RMSE: 0.4876462542360475


In [13]:
from sklearn.ensemble import GradientBoostingRegressor

gb_param_grid = {
    'n_estimators': [150, 300, 500],
    'learning_rate': [0.03, 0.05],
    'max_depth': [4, 5],
    'min_samples_leaf': [2, 3]
}

gb_grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Gradient Boosting")
gb_grid_search.fit(X_train, y_train)
print(gb_grid_search.best_params_)
best_gb_model = gb_grid_search.best_estimator_
y_pred_best_gb = best_gb_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_gb))
print("MAE: ", mean_absolute_error(y_test, y_pred_best_gb))
print("MSE: ", mean_squared_error(y_test, y_pred_best_gb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best_gb)))

Gradient Boosting
Fitting 3 folds for each of 24 candidates, totalling 72 fits
{'learning_rate': 0.03, 'max_depth': 4, 'min_samples_leaf': 2, 'n_estimators': 150}
R2: 0.5354522458560234
MAE:  0.34212116345872107
MSE:  0.2300125719682344
RMSE: 0.4795962593351145


При предыдущей сетке градиентному бустингу не хватает мощности, добавим больше деревьев. В гр. бустинге деревья строятся последовательно и исправляют ошибки друг друга. Модель остановилась на 4, так что поставим 5. 6 и 8 глубина не нужна. Так же изменим шаг lr чтобы вклад дерева был плавнее. UPD: помогло

In [11]:
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV

cb_param_grid = {
    'iterations': [150, 300],
    'learning_rate': [0.03, 0.05],
    'depth': [4, 6]
}

cb_grid_search = GridSearchCV(
    estimator=CatBoostRegressor(random_state=42, thread_count=-1, verbose=0),
    param_grid=cb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=1
)

print("CatBoost Regressor")
cb_grid_search.fit(X_train, y_train)
print(cb_grid_search.best_params_)
best_cb_model = cb_grid_search.best_estimator_
y_pred_best_cb = best_cb_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_cb))
print("MAE: ", mean_absolute_error(y_test, y_pred_best_cb))
print("MSE: ", mean_squared_error(y_test, y_pred_best_cb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best_cb)))

CatBoost Regressor
Fitting 3 folds for each of 8 candidates, totalling 24 fits
{'depth': 4, 'iterations': 300, 'learning_rate': 0.03}
R2: 0.4874381242895941
MAE:  0.3759433686398051
MSE:  0.2537859117245322
RMSE: 0.5037716861084317


Улучшить CatBoost не получается, деревья получаются либо специфичные либо идет переобучение, а если делать щадяще то результат средний. Стоит попробовать сделать анализ важности признаков

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(rf_search.best_params_)
best_rf_model = rf_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

mae_log = mean_absolute_error(y_test, y_pred_best_rf)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
r2 = r2_score(y_test, y_pred_best_rf)

y_test_real = 10 ** y_test
y_pred_real = 10 ** y_pred_best_rf
mae_real = mean_absolute_error(y_test_real, y_pred_real)

print("R2:",r2)
print("MAE",mae_log)
print("RMSE",rmse_log)
print("MAE real",mae_real)

{'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 400}
R2: 0.5197265535867324
MAE 0.3395063134855194
RMSE 0.4876462542360475
MAE real 270.73071882326775


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(gb_grid_search.best_params_)
best_gb_model = gb_grid_search.best_estimator_
y_pred_best_gb = best_gb_model.predict(X_test)

mae_log_gb = mean_absolute_error(y_test, y_pred_best_gb)
rmse_log_gb = np.sqrt(mean_squared_error(y_test, y_pred_best_gb))
r2_gb = r2_score(y_test, y_pred_best_gb)

y_test_real = 10 ** y_test
y_pred_real_gb = 10 ** y_pred_best_gb
mae_real_gb = mean_absolute_error(y_test_real, y_pred_real_gb)

print("R2:", r2_gb)
print("MAE", mae_log_gb)
print("RMSE", rmse_log_gb)
print("MAE real", mae_real_gb)

{'learning_rate': 0.03, 'max_depth': 4, 'min_samples_leaf': 2, 'n_estimators': 150}
R2: 0.5354522458560234
MAE 0.34212116345872107
RMSE 0.4795962593351145
MAE real 284.3581267585104


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(cb_grid_search.best_params_)
best_cat_model = cb_grid_search.best_estimator_
y_pred_best_cat = best_cat_model.predict(X_test)

mae_log_cat = mean_absolute_error(y_test, y_pred_best_cat)
rmse_log_cat = np.sqrt(mean_squared_error(y_test, y_pred_best_cat))
r2_cat = r2_score(y_test, y_pred_best_cat)

y_test_real = 10 ** y_test
y_pred_real_cat = 10 ** y_pred_best_cat
mae_real_cat = mean_absolute_error(y_test_real, y_pred_real_cat)

print("R2:", r2_cat)
print("MAE", mae_log_cat)
print("RMSE", rmse_log_cat)
print("MAE real", mae_real_cat)

{'depth': 4, 'iterations': 300, 'learning_rate': 0.03}
R2: 0.4874381242895941
MAE 0.3759433686398051
RMSE 0.5037716861084317
MAE real 313.7201004919353


Random Forest:<br>
R2: 0.5197265535867324<br>
MAE 0.3395063134855194<br>
RMSE 0.4876462542360475<br>
MAE 270.73071882326775<br>

Gradient Boosting:<br>
R2: 0.5354522458560234<br>
MAE 0.34212116345872107<br>
RMSE 0.4795962593351145<br>
MAE real 284.3581267585104<br>

CatBoost:<br>
R2: 0.5049508359484075<br>
MAE 0.34169197478571617<br>
RMSE 0.49509070229793245<br>
MAE real 268.34889079827275<br>

#Вывод
У Gradient Boosting самый высокий R2 = 0.5348, но при этом самая большая ошибка в реальных миллимолях MAE real = 284.36 mM.У CatBoost R2 ниже 0.5049, но в реальных миллимолях он ошибается меньше всего MAE real = 268.34 mM. Почему? R2 коэффициент оценивает общую точность модели по всему датасету в логарифмической шкале. Градиентный бустинг точнее предсказывает основную массу молекул. Но при обратном преобразовании даже микроскопическая ошибка на молекулах с высокой концентрацией превращается в огромную ошибку в миллимолях, таким образом CatBoost и Random Forest оказались более устойчивыми к выбросам, поэтому их реальная физическая ошибка получилась меньше. Но какую то из них выбрать достаточно трудно, остановимся на RF, так как представляется как наиболее сбалансированное решение